In [ ]:
import sys
from pathlib import Path

import torch
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration, CLIPProcessor
from PIL import Image
import os
import json
import torch.nn.functional as F
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

ROOT = Path.cwd()
if not (ROOT / "visionguard").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

from visionguard.critic.custom_clip import CustomCLIPModel


import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def set_trainable_parameters(model, trainable_substrings=None, matching_fn=None):
    """ Freeze all parameters in the model, and then selectively unfreeze specific parts. """
    for param in model.parameters():
        param.requires_grad = False

    def is_trainable(name):
        if trainable_substrings and any(sub in name for sub in trainable_substrings):
            return True
        if matching_fn and matching_fn(name):
            return True
        return False

    for name, param in model.named_parameters():
        if is_trainable(name):
            param.requires_grad = True
            # print(f"Unfrozen: {name}")


def extract_assistant_content(caption, initial_user_prompt="Describe this image."):
    """ 
    Extract the assistant's response from a caption. 
    Remove both the initial user prompt and 'ASSISTANT:' tag if they exist.
    """
    # Remove "ASSISTANT:" if it exists
    if "ASSISTANT:" in caption:
        caption = caption.split("ASSISTANT:")[-1].strip()
    
    # Remove the initial user prompt if it appears at the start of the caption
    if caption.startswith(initial_user_prompt):
        caption = caption[len(initial_user_prompt):].strip()

    return caption


def test_time_adaptation_for_all_images(
    image_directory,
    start_idx=0,
    end_idx=1004,
    initial_user_prompt="Describe this image.",
    model_path="Salesforce/instructblip-vicuna-7b",
    clip_model_path="../../clipmodel.pth",
    device_blip="cuda:0",
    device_clip="cuda:0",
    lr=2e-3,
    num_steps=5,
    num_beams=5,
    max_new_tokens=512
):
    image_paths = [os.path.join(image_directory, img) for img in os.listdir(image_directory) if img.endswith(('.jpg', '.jpeg', '.png'))]
    image_ids = [int(os.path.splitext(os.path.basename(img))[0].split('_')[-1]) for img in image_paths]
    image_paths_and_ids = sorted(zip(image_ids, image_paths), key=lambda x: x[0])  # Sort by image id

    # Apply the range from start_idx to end_idx
    image_paths_and_ids = image_paths_and_ids[start_idx:end_idx]
    print(f"Processing images from index {start_idx} to {end_idx}... Total: {len(image_paths_and_ids)} images")
    # Dynamically set the output file names
    intermediate_results_json = f"tta_intermediate_results_{start_idx}_{end_idx}.json"
    baseline_results_json = f"tta_baseline_results_{start_idx}_{end_idx}.json"

    # image_paths_and_ids = image_paths_and_ids[:250]  # Limit to first 3 images for testing
    # Store intermediate and final baseline results
    intermediate_results_data = []
    baseline_results_data = []

    for idx, (image_id, image_path) in enumerate(tqdm(image_paths_and_ids, desc="Processing Images")):
        # print(f"\nProcessing image {image_path} (ID {image_id}) [{idx+1}/{len(image_paths_and_ids)}]")

        # Clear GPU cache to avoid memory issues
        torch.cuda.empty_cache()

        # Load models fresh from checkpoint

        instructblip_model = InstructBlipForConditionalGeneration.from_pretrained(model_path).to(device_blip)
        instructblip_processor = InstructBlipProcessor.from_pretrained(model_path)


        # Load Custom CLIP Model
        clip_model = CustomCLIPModel()
        state_dict = torch.load(clip_model_path)
        clip_model.load_state_dict(state_dict)
        clip_model.to(device_clip)
        clip_model.eval()

        instructblip_model.train()
        set_trainable_parameters(instructblip_model, matching_fn=lambda n: "language" in n and "norm" in n)


        # set_trainable_parameters(llava_model, matching_fn=lambda n: "projector" in n)

        optimizer = AdamW(filter(lambda p: p.requires_grad, instructblip_model.parameters()), lr=lr)

        image_results = {
            "image_id": image_id,
            "image_path": image_path,
            "initial_prompt": initial_user_prompt,
            "steps": []
        }

        image = Image.open(image_path).convert("RGB")

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": initial_user_prompt},
                ],
            }
        ]

        for step in range(num_steps):
            # print(f"Step {step+1}/{num_steps} starting...")

            inputs = instructblip_processor(images=image, text=initial_user_prompt, return_tensors="pt").to(device_blip)
            # print("Inputs: ", inputs)
            # print(inputs.keys())
            # Check if 'qformer_input_ids' and 'qformer_attention_mask' are in inputs
            # if 'qformer_input_ids' in inputs and 'qformer_attention_mask' in inputs:
            #     # Duplicate (repeat) qformer_input_ids and qformer_attention_mask to match num_beams
            #     # Assuming the original batch size is 1, we repeat it 'num_beams' times
            #     inputs['qformer_input_ids'] = inputs['qformer_input_ids'].repeat(num_beams, 1)
            #     inputs['qformer_attention_mask'] = inputs['qformer_attention_mask'].repeat(num_beams, 1)
            # else:
            #     raise KeyError("Required keys 'qformer_input_ids' or 'qformer_attention_mask' not found in inputs.")

            # print("inputs['qformer_input_ids'].shape: ", inputs['qformer_input_ids'].shape)
            # Call generate() with required arguments only
            generated_ids = instructblip_model.generate(
                **inputs,
                do_sample=False,
                num_beams=num_beams,
                max_length=77,
                num_return_sequences=num_beams,  # Ensure it matches num_beams
            )
            candidates = instructblip_processor.batch_decode(generated_ids, skip_special_tokens=True)
            # print("Candidates: ", candidates)
            rewards = []
            assistant_contents = [extract_assistant_content(c) for c in candidates]  
            # print("Candidates: ", assistant_contents)

            for candidate in assistant_contents:
                clip_inputs = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")(
                    text=[candidate], 
                    images=image, 
                    return_tensors="pt", 
                    truncation=True, 
                    padding="max_length"
                ).to(device_clip)

                image_inputs = clip_inputs['pixel_values']
                text_inputs = {'input_ids': clip_inputs['input_ids'], 'attention_mask': clip_inputs['attention_mask']}

                with torch.no_grad():
                    _, logits_binary_classification, clip_scores = clip_model.infer(image_inputs, text_inputs)

                clip_score = clip_scores.squeeze(0)
                logits_binary_classification = logits_binary_classification.squeeze(0)

                rewards.append((clip_score.item(), logits_binary_classification.item()))

            # Extract clip_scores and logit scores
            clip_scores, logit_scores = zip(*rewards)
            clip_scores = torch.tensor(clip_scores)
            logit_scores = torch.tensor(logit_scores)

            clip_score_mean = clip_scores.mean()
            logit_score_mean = logit_scores.mean()

            normalized_clip_scores = clip_scores - clip_score_mean
            normalized_logits = logit_scores - logit_score_mean

            combined_rewards = normalized_clip_scores + normalized_logits
            rewards = combined_rewards.to(device_blip, torch.float32)

            # assistant_contents = [extract_assistant_content(c) for c in candidates]
            assistant_contents = candidates
            # print("Assistant Contents: ", assistant_contents)
            step_data = {
                "step": step+1,
                "beam_captions": assistant_contents,
                "clip_scores": clip_scores.tolist(),
                "rewards": rewards.tolist()
            }
            image_results["steps"].append(step_data)

            if step == 0:
                best_idx_step1 = clip_scores.argmax().item()
                best_caption_step1 = assistant_contents[best_idx_step1]
                best_score_step1 = clip_scores[best_idx_step1].item()
                image_results["initial_best_caption"] = best_caption_step1
                image_results["initial_best_score"] = best_score_step1
            # print("image shape: ", image.size)
            # print("text shape: ", len(assistant_contents))
            # Compute log probs for RL update
            total_loss = 0
            num_beams = len(assistant_contents)  # Assume assistant_contents corresponds to num_beams
            # print("assistant_contents: ", assistant_contents)
            for i, content in enumerate(assistant_contents):
                # Prepare single beam input
                single_beam_inputs = instructblip_processor(images=image, text=content, return_tensors="pt", padding=True).to(device_blip)
                
                # Run model for this single beam (batch size is now 1)
                outputs = instructblip_model(
                    **single_beam_inputs,
                    return_dict=True
                )

                logits = outputs.logits  # Shape (1, seq_len, vocab_size) since batch size is 1
                logits_float = logits.float()  # Convert to float to avoid issues with mixed precision
                log_probs = F.log_softmax(logits_float, dim=-1)  # Log probabilities for token generation

                # Get log probs for actual tokens (use input_ids from the single beam)
                input_ids = single_beam_inputs["input_ids"]  # Shape: (1, seq_len)
                log_probs_for_tokens = torch.gather(log_probs, 2, input_ids.unsqueeze(-1)).squeeze(-1)  # Shape: (1, seq_len)
                log_probs_sum = log_probs_for_tokens.sum(dim=1)  # Sum the log probabilities for the sequence, Shape: (1,)
                
                # Use reward from rewards (rewards[i] should correspond to this beam)
                reward = rewards[i].to(device_blip)  # Make sure the reward is on the correct device
                loss = -(reward * log_probs_sum).mean()  # Compute the loss for this beam

                total_loss += loss  # Accumulate the loss

            # Average loss over beams
            final_loss = total_loss / num_beams

            # Backpropagation
            optimizer.zero_grad()
            final_loss.backward()
            clip_grad_norm_(instructblip_model.parameters(), max_norm=1.0)
            optimizer.step()

        final_inputs = instructblip_processor(images=image, text=initial_user_prompt, return_tensors="pt").to(device_blip)

        with torch.no_grad():
            final_generated = instructblip_model.generate(
                **final_inputs,
                max_length=max_new_tokens
            )

        # final_caption = extract_assistant_content(instructblip_processor.batch_decode(final_generated, skip_special_tokens=True)[0])
        final_caption = instructblip_processor.batch_decode(final_generated, skip_special_tokens=True)[0]
        image_results["final_caption"] = final_caption
        # print(f"Image ID: {image_id}, Final Refined Caption: {final_caption}\n")
        baseline_results_data.append({
            "id": image_id,
            "response": final_caption
        })

        intermediate_results_data.append(image_results)
        # print(f"Image ID: {image_id}, Final Refined Caption: {final_caption}\n")


        with open(intermediate_results_json, "w") as f:
            json.dump(intermediate_results_data, f, indent=4)

        with open(baseline_results_json, "w") as f:
            json.dump(baseline_results_data, f, indent=4)

        # print(f"Intermediate results saved to {intermediate_results_json}")
        # print(f"Baseline results saved to {baseline_results_json}")



# Example usage
image_directory = "../../data/AMBER/image"

test_time_adaptation_for_all_images(
    image_directory=image_directory,
    start_idx=0, 
    end_idx=250,
    model_path="Salesforce/instructblip-vicuna-7b",
    clip_model_path="../../clipmodel.pth",
    device_blip="cuda:3",
    device_clip="cuda:3",
    lr=2e-4,
    num_steps=5,
    num_beams=5,
    max_new_tokens=512
)